# 25.1 — Neurosymbolic integration

**Lesson tagline.** Neurosymbolic systems keep neural uncertainty, but ask it to pay a real price when it violates a rule.

Neural probabilities become soft truth values, binary cross-entropy remains the data term, and differentiable logic adds a penalty for violating rules. The key question is how much symbolic structure to inject without letting it dominate the data.

Save a copy to Drive to edit.

In [ ]:

import math
import itertools
import random
from collections import defaultdict
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt

SEED = 25
random.seed(SEED)
np.random.seed(SEED)


def sigmoid(z):
    z = np.asarray(z, dtype=float)
    return 1.0 / (1.0 + np.exp(-np.clip(z, -40.0, 40.0)))


def binary_cross_entropy(p, y):
    p = np.clip(np.asarray(p, dtype=float), 1e-12, 1.0 - 1e-12)
    y = np.asarray(y, dtype=float)
    terms = -y * np.log(p) - (1.0 - y) * np.log(1.0 - p)
    return float(np.mean(terms))



def soft_implication(A, B):
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)
    return np.maximum(1.0 - A, B)


def probabilistic_or(A, B):
    return A + B - A * B


def product_and(A, B):
    return A * B


def neurosymbolic_loss(p, y, A, B, lambda_):
    neural = binary_cross_entropy(p, y)
    satisfaction = soft_implication(A, B)
    penalty = float(np.mean(1.0 - satisfaction))
    total = neural + lambda_ * penalty
    return total, neural, penalty


def make_neurosymbolic_ladder(seed=25):
    rng = np.random.default_rng(seed)
    rungs = []
    p1 = np.array([0.20, 0.70, 0.90])
    y1 = np.array([0, 1, 1])
    A1 = np.array([0.2, 0.7, 0.9])
    B1 = np.array([0.8, 0.6, 0.4])
    rungs.append({'name': 'D1 lesson rule toy', 'p': p1, 'y': y1, 'A': A1, 'B': B1, 'lambda': 2.0})
    for n, noise, conflict, name in [
        (24, 0.00, 0.00, 'D2 clean implication classifier'),
        (48, 0.08, 0.15, 'D3 noisy labels with conflicts'),
        (80, 0.10, 0.25, 'D4 real-ish monotone risk subset'),
        (140, 0.18, 0.42, 'D5 shifted noisy conflict stress test'),
    ]:
        x = rng.normal(size=(n, 2))
        logits = 1.4 * x[:, 0] - 0.8 * x[:, 1]
        y = (logits > 0.0).astype(float)
        flip = rng.random(n) < noise
        y[flip] = 1.0 - y[flip]
        A = sigmoid(1.5 * x[:, 0] + 0.4 * rng.normal(size=n))
        B = sigmoid(1.2 * x[:, 0] - 0.6 * x[:, 1] + 0.4 * rng.normal(size=n))
        conflict_ix = rng.choice(n, size=int(conflict * n), replace=False)
        B[conflict_ix] = 1.0 - B[conflict_ix]
        p = sigmoid(logits + 0.7 * rng.normal(size=n))
        rungs.append({'name': name, 'p': p, 'y': y, 'A': A, 'B': B, 'lambda': 2.0})
    return rungs


## The concept, built once (D1)

We combine the lesson's neural BCE with a soft implication penalty. The rule formula is $A\Rightarrow B=\max(1-A,B)$ and the objective is $L_{total}=L_{neural}+\lambda(1-S_{rule})$.

In [ ]:

p = np.array([0.20, 0.70, 0.90])
y = np.array([0, 1, 1])
A = np.array([0.2, 0.7, 0.9])
B = np.array([0.8, 0.6, 0.4])
lesson_bce = binary_cross_entropy(p, y)
lesson_penalties = 1.0 - soft_implication(A, B)
print('BCE:', lesson_bce)
print('implication penalties:', lesson_penalties)
assert abs(lesson_bce - 0.22839300363692283) < 1e-15
assert np.allclose(lesson_penalties, np.array([0.2, 0.4, 0.6]))


The rule weight is not harmless. With the middle case penalty $0.4$, the lesson totals are $0.22839300363692283+\lambda\cdot0.4$.

In [ ]:

for lambda_ in [0.0, 0.5, 1.0, 2.0, 5.0]:
    total = lesson_bce + lambda_ * 0.4
    print(lambda_, total)
assert abs(lesson_bce + 0.5 * 0.4 - 0.42839300363692284) < 1e-15
assert abs(lesson_bce + 2.0 * 0.4 - 1.0283930036369228) < 1e-15
assert abs(lesson_bce + 5.0 * 0.4 - 2.228393003636923) < 1e-15
assert abs(product_and(0.70, 0.60) - 0.42) < 1e-15
assert abs(probabilistic_or(0.70, 0.60) - 0.88) < 1e-15


## The dataset ladder

The instance-size ladder moves from the three lesson probabilities to larger clean, noisy, real-ish, and shifted/conflicting rule datasets.

In [ ]:

rungs = make_neurosymbolic_ladder()
for i, rung in enumerate(rungs, start=1):
    conflict_rate = float(np.mean(1.0 - soft_implication(rung['A'], rung['B'])))
    print(f"D{i}: {rung['name']} n={len(rung['y'])} positive_rate={np.mean(rung['y']):.3f} rule_penalty={conflict_rate:.3f}")
    print('sample p,y,A,B:', list(zip(rung['p'][:3], rung['y'][:3], rung['A'][:3], rung['B'][:3])))


## Run the SAME method across D1–D5

In [ ]:

results = []
for i, rung in enumerate(rungs, start=1):
    total, neural, penalty = neurosymbolic_loss(rung['p'], rung['y'], rung['A'], rung['B'], rung['lambda'])
    results.append({'D': i, 'name': rung['name'], 'total': total, 'neural': neural, 'penalty': penalty})
print('rung | total_loss | neural_loss | rule_penalty')
for row in results:
    print(f"D{row['D']} | {row['total']:.4f} | {row['neural']:.4f} | {row['penalty']:.4f}")


## Results visualization

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for col, rung in enumerate(rungs):
    axes[0, col].scatter(rung['A'], rung['B'], c=rung['y'], cmap='coolwarm', s=20)
    axes[0, col].plot([0, 1], [1, 0], color='black', linewidth=1)
    axes[0, col].set_title(f"D{col + 1} rule space")
    axes[0, col].set_xlabel('A')
    axes[0, col].set_ylabel('B')
axes[1, 0].axis('off')
axes[1, 1].plot([row['D'] for row in results], [row['total'] for row in results], marker='o', label='total')
axes[1, 1].plot([row['D'] for row in results], [row['neural'] for row in results], marker='o', label='neural')
axes[1, 1].plot([row['D'] for row in results], [row['penalty'] for row in results], marker='o', label='rule')
axes[1, 1].set_title('loss components vs complexity')
axes[1, 1].set_xlabel('rung')
axes[1, 1].legend()
for ax in axes[1, 2:]:
    ax.axis('off')
plt.tight_layout()
plt.show()


## Pitfall on the hardest rung

On D5, a high $\lambda$ can make the rule dominate the data. We reproduce that domination, then sweep validation-style weights and report components instead of hiding them.

In [ ]:

d5 = rungs[-1]
for lambda_ in [0.0, 0.5, 2.0, 10.0]:
    total, neural, penalty = neurosymbolic_loss(d5['p'], d5['y'], d5['A'], d5['B'], lambda_)
    print(f"lambda={lambda_:>4}: total={total:.4f} neural={neural:.4f} rule={penalty:.4f}")
sweep = []
for lambda_ in np.linspace(0.0, 4.0, 9):
    total, neural, penalty = neurosymbolic_loss(d5['p'], d5['y'], d5['A'], d5['B'], lambda_)
    balanced = total + 0.25 * abs(neural - penalty)
    sweep.append((balanced, lambda_, total, neural, penalty))
best = min(sweep)
print('chosen lambda from component-aware sweep:', best[1])
print('balanced,total,neural,rule:', best[0], best[2], best[3], best[4])


## Evaluate it + Practice

- **Metric.** Track total loss with neural and rule components, and compare it with a neural-only model with lambda zero.
- **Cheap sanity check.** D1 must reproduce BCE 0.22839300363692283 and penalties 0.2, 0.4, 0.6.
- **Ablation.** turn off the rule penalty and observe violation rise on conflict-heavy rungs.
- **Failure signals.** large lambda lowers rule violations while neural loss becomes unacceptable.

### Practice prompts


- Change the implication relaxation and compare the D5 sweep.

- Add a second rule and plot both rule penalties separately.

- Design a validation criterion that rejects rule domination.